# 📓 EDA 탐색 노트북 — 상품군별 템플릿

`docs/EDA_GUIDE.md` **§3의 질문 11개**를 푸는 **작업장**입니다.

## 쓰는 법

1. 이 파일을 `notebooks/<domain>_<이름>.ipynb` 로 **복사**해서 쓰세요 (원본은 그대로 두기)
2. 아래 `DOMAIN` 을 담당 테이블로 바꾸고 위에서부터 실행
3. 질문별 셀에서 자유롭게 탐색 — 셀 추가/삭제 마음대로
4. **결론은 `docs/eda/<domain>_notes.md` 로 옮겨 적으세요** ← 이게 산출물입니다
5. 마지막 절에서 질의 20개 검증 + `eval/questions_<domain>.jsonl` 자동 생성

## ⚠️ 이 노트북은 산출물이 아닙니다

`.ipynb` 는 JSON + 출력셀 + 실행카운터라 **git diff 가 안 읽혀 리뷰가 불가능**하고,
탐색 과정이 전부 남아 **결론이 안 보입니다**.

> **노트북 = 작업장 / `_notes.md` + `.jsonl` = 산출물**

개인 노트북은 `.gitignore` 되어 있습니다 (이 템플릿만 커밋됨).

## 🔒 원본 DB 훼손 방지

다음 셀에서 **읽기 전용 커넥션**을 만들고, 쓰기가 실제로 막혔는지 확인합니다.
노트북은 셀을 순서 없이 재실행하는 게 자연스러워서 스크립트보다 위험합니다 —
`df.to_sql(...)` 오타 하나로 테이블이 날아갈 수 있습니다. 이 셀을 **반드시 먼저** 실행하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 안전 셋업 — 반드시 먼저 실행
# ═══════════════════════════════════════════════════════════
import sqlite3, json, sys, collections
from pathlib import Path
import pandas as pd

# ← 담당 테이블로 변경
DOMAIN = "domestic_etfs"
#   domestic_bonds / domestic_etfs / overseas_etfs / public_funds

ROOT = Path.cwd()
if not (ROOT / "data").exists():          # notebooks/ 안에서 열었을 때
    ROOT = ROOT.parent
DB = (ROOT / "data" / "financial_products.db").resolve()
assert DB.exists(), f"DB 없음: {DB}\n먼저 `python scripts/build_db.py` 실행"

# 🔒 읽기 전용 커넥션 — 쓰기 시도는 커넥션 레벨에서 거부됩니다
conn = sqlite3.connect(f"{DB.as_uri()}?mode=ro", uri=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

# 쓰기가 실제로 막혔는지 확인 (안 막혔으면 여기서 멈춤)
try:
    conn.execute("CREATE TABLE __probe__(x)")
    raise RuntimeError("❌ 쓰기가 허용됩니다 — 커넥션 설정을 확인하세요!")
except sqlite3.OperationalError as e:
    assert "readonly" in str(e), e
    print(f"🔒 읽기 전용 확인 — 원본 훼손 불가 ({e})")

TOTAL = conn.execute(f"SELECT COUNT(*) FROM {DOMAIN}").fetchone()[0]
COLS = [r[1] for r in conn.execute(f'PRAGMA table_info("{DOMAIN}")')]
print(f"📊 {DOMAIN}: {TOTAL:,}행 × {len(COLS)}컬럼")

## 헬퍼

`TRIM()` 이 기본으로 적용됩니다 — §8-① 패딩 함정 때문에 **맨손 정확일치는 조용히 실패**합니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 자주 쓰는 헬퍼
# ═══════════════════════════════════════════════════════════

def q(sql, params=()):
    """SQL 실행 → DataFrame"""
    return pd.read_sql_query(sql, conn, params=params)


def vals(col, table=None, limit=50):
    """distinct 값 + 건수 (TRIM 적용). 범주형 컬럼 파악용."""
    t = table or DOMAIN
    return q(f"""SELECT TRIM(CAST("{col}" AS TEXT)) AS value, COUNT(*) AS n
                 FROM {t}
                 WHERE TRIM(COALESCE(CAST("{col}" AS TEXT), '')) <> ''
                 GROUP BY 1 ORDER BY n DESC LIMIT {limit}""")


def miss(col, table=None):
    """결측 3종 내역 — NULL / 공백문자열 / 패딩 (§8-②)"""
    t = table or DOMAIN
    return q(f"""SELECT COUNT(*) AS total,
                   SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_null,
                   SUM(CASE WHEN "{col}" IS NOT NULL
                             AND TRIM(CAST("{col}" AS TEXT))='' THEN 1 ELSE 0 END) AS n_blank,
                   SUM(CASE WHEN CAST("{col}" AS TEXT)
                             <> TRIM(CAST("{col}" AS TEXT)) THEN 1 ELSE 0 END) AS n_padded
                 FROM {t}""")


def peek(col, n=10, table=None):
    """실제 저장값 샘플. 패딩이 눈에 보이도록 repr 로 출력."""
    t = table or DOMAIN
    rows = q(f'SELECT "{col}" AS v FROM {t} WHERE "{col}" IS NOT NULL LIMIT {n}')
    for v in rows["v"]:
        print(repr(v))


def cross(col, other_table, other_col):
    """
    §3 질문 5 — 우리 테이블의 값이 다른 테이블과 같은 표기를 쓰는지.
    겹치는 값 / 우리만 있는 값 / 상대만 있는 값을 보여줍니다.
    """
    a = set(vals(col, limit=100000)["value"])
    b = set(vals(other_col, table=other_table, limit=100000)["value"])
    print(f"{DOMAIN}.{col}: {len(a)}종   {other_table}.{other_col}: {len(b)}종")
    print(f"  ✅ 겹침      {len(a & b):>5}종  {sorted(a & b)[:8]}")
    print(f"  ◀ 우리만    {len(a - b):>5}종  {sorted(a - b)[:8]}")
    print(f"  ▶ 상대만    {len(b - a):>5}종  {sorted(b - a)[:8]}")
    return a, b


def kor(col):
    """한글 컬럼명"""
    r = q("SELECT korean_name FROM schema_metadata WHERE table_name=? AND column_name=?",
          (DOMAIN, col))
    return r["korean_name"][0] if len(r) else ""


# 결측 후보 패턴은 프로파일러가 단일 정의를 갖습니다 — 노트북에서 다시 정의하지 않습니다.
sys.path.insert(0, str(ROOT / "scripts"))
from profile_table import NOT_PROVIDED_PATTERNS, NOT_APPLICABLE_PATTERNS   # noqa: E402


# ── 이 작업의 근거 (docs/research/notes/) ──────────────────────
# 왜 이 작업을 하는지가 코드 실행 결과에도 남도록 인용을 들고 다닙니다.
# 확인 수준을 함께 적어, 아직 원문을 안 읽은 근거를 과신하지 않게 합니다.
RESEARCH = {
    "semantic_layer": (
        "arXiv 2604.25149 — 스키마만 45.5~50.5% vs 시맨틱 레이어 추가 67.7~68.7% "
        "(+17~23%p, p<0.01). 모델 선택 차이는 미미하고 레이어가 분산의 대부분을 설명.",
        "초록만", "semantic-layer-benchmark.md"),
    "fibo_class": (
        "FIBO CollectiveInvestmentVehicles.rdf — 펀드 유형(EquityFund 등)은 owl:Class "
        "하위클래스이지 개체가 아님. 클래스(종류형)는 FundShareClassUnit 독립 개체.",
        "부분 확인", "FIBO_funds-model.md"),
}


def cite(key):
    """근거를 실행 결과에 남긴다. 확인 수준을 반드시 함께 출력."""
    msg, level, doc = RESEARCH[key]
    print(f"📚 {msg}")
    print(f"   └ 확인 수준: {level} · docs/research/notes/{doc}")


print("헬퍼: q() vals() miss() peek() cross() kor() cite() / overview() 는 다음 셀")
print(f"결측 후보 패턴 (profile_table 에서 import): "
      f"not_provided {len(NOT_PROVIDED_PATTERNS)}종 · not_applicable {len(NOT_APPLICABLE_PATTERNS)}종")

## 전체 개요 — 컬럼 전수 기초 통계

`overview()` 로 **모든 컬럼의 기초 통계를 한 표**로 봅니다.
DataFrame 이므로 정렬·필터가 자유롭습니다.

```python
overview()                                # 컬럼명 순
overview('결측률')                          # 결측 많은 순  ← 결측치 확인은 여기서
overview('distinct')                       # 카디널리티 순
overview().query("kind=='numeric'")        # 수치형만
overview().query("판정대기 > 0")             # 결측 판정이 필요한 컬럼만
```

> ⚠️ **`결측률` 은 참고용 보조 지표입니다.** `null + 공백` 만으로 계산한 값이고,
> `판정대기` 열의 값이 결측인지 정보인지는 **여러분이 정합니다** (EDA_GUIDE §3 🕳️).

In [ ]:
# ═══════════════════════════════════════════════════════════
# 컬럼 전수 개요
# ═══════════════════════════════════════════════════════════
import yaml

auto_path = ROOT / "ontology" / "enums" / f"{DOMAIN}.auto.yaml"
assert auto_path.exists(), f"{auto_path.name} 없음 — `python scripts/profile_table.py {DOMAIN}` 먼저 실행"
AUTO = yaml.safe_load(auto_path.read_text(encoding="utf-8"))


def overview(sort="column", ascending=None):
    """모든 컬럼의 기초 통계를 한 DataFrame 으로. .auto.yaml 을 읽으므로 즉시 반환."""
    rows = []
    for c, e in AUTO["columns"].items():
        n = e.get("numeric") or {}
        if n:
            rep = f"{n['min']:,.1f} ~ {n['max']:,.1f}"
        elif e.get("values"):
            rep = " · ".join(str(v["value"])[:12] for v in e["values"][:3])
        elif e.get("values_top"):
            rep = " · ".join(str(v["value"])[:12] for v in e["values_top"][:2]) + " …"
        else:
            rep = ""
        rows.append(dict(
            column=c, 한글명=(e["korean_name"] or "")[:18], kind=e["kind"],
            값있음=e["values_present"], null=e["null"], 공백=e["blank_string"],
            결측률=round(1 - e["values_present"] / TOTAL, 3),
            distinct=e["distinct_count"],
            판정대기=sum(j["count"] for j in e.get("judgment_needed", [])),
            대표값=rep,
        ))
    df = pd.DataFrame(rows)
    if ascending is None:
        ascending = (sort == "column")
    return df.sort_values(sort, ascending=ascending).reset_index(drop=True)


print(f"{DOMAIN}: {TOTAL:,}행 × {len(AUTO['columns'])}컬럼\n")
display(overview("결측률").head(15))

---

# 🕳️ 결측 판정 — 채우지 말고 분류하기

> **이 단계를 먼저 끝내고 나머지 탐색으로 갑니다.**
> 결측이 무엇인지 모른 채 분포를 보면 그 분포가 왜곡돼 있습니다.

**대치(impute)하지 않습니다.** 없는 값을 만들면 그게 환각입니다.
여기서 하는 일은 **각 컬럼의 결측이 왜 없는지 판정하고 라벨을 붙이는 것**입니다.

| 유형 | 뜻 | 런타임 응답 |
| :--- | :--- | :--- |
| `not_applicable` | 그 대상엔 원래 해당 없음 | "해당사항 없습니다" |
| `missing` | 제공자가 안 줌 / 있어야 하는데 없음 | "데이터 미제공" · "확인할 수 없음" |

**판정 방법:** 결측이 **다른 컬럼과 대응하는지** 봅니다. 대응하면 구조적(=해당없음)입니다.

> 🔑 **판정과 규칙은 노트북이 아니라 `ontology/enums/<domain>.yaml` 에 삽니다.**
> 노트북은 gitignore 대상이라 여기 적어두면 런타임에도 팀원에게도 전달되지 않습니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1) 결측이 있는 컬럼 나열 — 판정 대상
# ═══════════════════════════════════════════════════════════
miss_cols = overview("결측률").query("결측률 > 0")
print(f"결측이 있는 컬럼 {len(miss_cols)}개 / 전체 {len(AUTO['columns'])}개")
print("→ 각 컬럼의 결측이 '해당없음'인지 '미제공'인지가 그대로 답변 분기가 됩니다.\n")
display(miss_cols[["column", "한글명", "kind", "값있음", "null", "결측률", "판정대기"]])

## 원인 규명 — 결측이 다른 컬럼과 대응하는가

한쪽으로 쏠리면 **구조적 결측(해당없음)** 일 가능성이 높습니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2) 결측 원인 교차 확인
# ═══════════════════════════════════════════════════════════

def why_missing(col, by):
    """`col` 의 결측이 `by` 값에 따라 갈리는지 본다."""
    df = q(f"""SELECT TRIM(CAST("{by}" AS TEXT)) AS 기준,
                 SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS 결측,
                 COUNT(*) AS 전체
               FROM {DOMAIN} GROUP BY 1 ORDER BY 3 DESC""")
    df["결측률"] = (df["결측"] / df["전체"]).round(3)
    return df


def co_missing(a, b):
    """두 컬럼이 함께 결측되는지 — 같은 원인인지 확인."""
    return q(f"""SELECT
        SUM(CASE WHEN "{a}" IS NULL AND "{b}" IS NULL THEN 1 ELSE 0 END) AS 둘다결측,
        SUM(CASE WHEN "{a}" IS NULL AND "{b}" IS NOT NULL THEN 1 ELSE 0 END) AS "{a}만",
        SUM(CASE WHEN "{a}" IS NOT NULL AND "{b}" IS NULL THEN 1 ELSE 0 END) AS "{b}만"
      FROM {DOMAIN}""")


# 예시 — 담당 컬럼에 맞게 바꿔 쓰세요
# display(why_missing("<결측컬럼>", "<기준컬럼>"))
# display(co_missing("<컬럼A>", "<컬럼B>"))

## 판정과 규칙 기록

세 가지를 정합니다. **전부 `<domain>.yaml` 로 나갑니다.**

| 블록 | 내용 | 런타임 소비처 |
| :--- | :--- | :--- |
| `columns[].missing_reason` | 컬럼 전체의 결측 성격 | 응답 분기 |
| `columns[].missing_semantics` | **특정 값**이 결측인지 정보인지 | 결측 계산 |
| `columns[].answer_policy` | 이 컬럼으로 답할 때의 규칙 | 답변 조립 |
| `normalization` | `TRIM` 필요 컬럼 등 | SQL 생성 |
| `query_rules` | 필수 필터·GROUP BY | SQL 생성 · `gold_sql` |

> 📌 판정이 안 서면 `missing` 으로 두세요. **"확인할 수 없음"이 틀린 답보다 안전합니다.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3) 판정 기록 — 근거를 함께 남긴다
# ═══════════════════════════════════════════════════════════
DECL_PATH = ROOT / "ontology" / "enums" / f"{DOMAIN}.yaml"
DECL = yaml.safe_load(DECL_PATH.read_text(encoding="utf-8")) if DECL_PATH.exists() else {}
print("이미 판정된 컬럼:", list((DECL.get("columns") or {}).keys()) or "없음")

# ── 새로 추가할 판정 ──
DECISIONS = {
    # "컬럼명": dict(
    #     missing_reason="not_applicable",          # 또는 "missing"
    #     note="근거 — 수치를 함께 (예: 국내 87.7% 결측 vs 해외 0.8%)",
    #     answer_policy="런타임이 이 컬럼으로 답할 때의 규칙",
    # ),
    # "값단위 판정이면":
    # "컬럼명": dict(missing_semantics={"해당없음": "not_applicable"}, note="근거"),
}

# ── 정규화 규칙 ──
TRIM_COLS = sorted({c for c, e in AUTO["columns"].items()
                    for f in e.get("findings", []) if f["detector"] == "padding"})

NORMALIZATION = {"trim_columns": TRIM_COLS}

# ⚠️ 기존 규칙 위에 **덧씌우기**. 통째로 대입하면 다른 경로로 추가된 규칙이 사라집니다.
QUERY_RULES = {
    **DECL.get("query_rules", {}),        # ← 기존 것을 먼저 싣는다
    # "종목단위": "GROUP BY <PK>",         # 행 중복이 있으면 필수
    # "판매중만": "<판매여부컬럼> = '...'",
}

print(f"\n추가 판정 {len(DECISIONS)}컬럼 · TRIM 대상 {len(TRIM_COLS)}컬럼 · 질의규칙 {len(QUERY_RULES)}개")
print(f"  기존 규칙 {len(DECL.get('query_rules', {}))}개를 유지한 채 병합합니다")

## `<domain>.yaml` 로 내보내기

**이 파일이 런타임 가드레일이 읽는 파일입니다.** 노트북 안에만 두면 시스템에 전달되지 않습니다.

내보낸 뒤 프로파일러를 다시 돌리면 `judgment_needed` 가 비워집니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4) yaml 내보내기 (기존 내용과 병합)
# ═══════════════════════════════════════════════════════════
DECL.setdefault("domain", DOMAIN)
DECL.setdefault("columns", {})

for col, d in DECISIONS.items():
    entry = DECL["columns"].setdefault(col, {})
    entry["korean_name"] = kor(col)
    for k in ("missing_reason", "missing_semantics", "answer_policy", "note"):
        if k in d:
            entry[k] = d[k]

# 병합 — 기존 키를 지우지 않습니다 (다른 경로로 추가된 규칙 보존)
if NORMALIZATION.get("trim_columns"):
    DECL.setdefault("normalization", {}).update(NORMALIZATION)
if QUERY_RULES:
    DECL.setdefault("query_rules", {}).update(QUERY_RULES)

# 왕복 검증 — 내보내기 전에 기존 키가 유실되지 않는지 확인
_before = set((yaml.safe_load(DECL_PATH.read_text(encoding="utf-8")) or {}).get("query_rules", {})) \
    if DECL_PATH.exists() else set()
_lost = _before - set(DECL.get("query_rules", {}))
assert not _lost, f"❌ 기존 query_rules 가 유실됩니다: {_lost}"

with DECL_PATH.open("w", encoding="utf-8") as fp:
    fp.write(f"# {DOMAIN} — 사람의 판단을 담는 파일\n")
    fp.write("# 기계적 사실은 *.auto.yaml 에 있습니다. 여기엔 판정과 규칙만 씁니다.\n")
    fp.write("# 런타임 가드레일이 이 파일을 읽습니다.\n\n")
    yaml.safe_dump(DECL, fp, allow_unicode=True, sort_keys=False, width=100)

print(f"✅ {DECL_PATH.relative_to(ROOT)}")
print(f"   판정 {len(DECL['columns'])}컬럼 · normalization {'있음' if 'normalization' in DECL else '없음'}"
      f" · query_rules {len(DECL.get('query_rules', {}))}개")

# 이 파일의 정체 — 실행 결과에 남긴다
print("\n📌 이 파일이 선행연구에서 말하는 '시맨틱 레이어'에 해당합니다.")
cite("semantic_layer")
print("   └ 논문의 4KB 손으로 쓴 마크다운 ≒ 이 yaml 의 answer_policy·normalization·query_rules")
print(f"\n다음: python scripts/profile_table.py {DOMAIN}   → judgment_needed 확인")

## 규칙 사용 — 정제 데이터를 만들지 않습니다

> ⚠️ **정제된 DataFrame 을 만들지 마세요.**
> 런타임 에이전트는 **원본 테이블**에 SQL 을 날립니다.
> 노트북에서만 깨끗한 데이터를 쓰면, 여기서 검증한 `gold_sql` 이 런타임에서 다른 결과를 냅니다.
> `gold_sql` 은 우리 유일한 정답 기준이라 여기가 어긋나면 평가셋 전체가 무의미해집니다.

규칙은 **SQL 조각**으로 들고 다니며 원본에 매번 적용합니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5) 규칙을 SQL 에 적용 — yaml 에서 로드해서 씀 (재정의 금지)
# ═══════════════════════════════════════════════════════════
RULES = DECL.get("query_rules", {})
TRIMS = set(DECL.get("normalization", {}).get("trim_columns", []))


def col(name):
    """규칙에 따라 컬럼 표현식을 만든다. 패딩 컬럼이면 TRIM 을 씌운다."""
    return f'TRIM("{name}")' if name in TRIMS else f'"{name}"'


for k, v in RULES.items():
    print(f"  {k:<10} {v}")
print(f"\nTRIM 대상 {len(TRIMS)}컬럼 — col('컬럼명') 으로 표현식 생성")
print("※ 이 규칙이 그대로 gold_sql 에 들어갑니다. 노트북과 런타임이 같은 SQL 을 봅니다.")

---

## 1단계 산출물 확인 — 프로파일러가 이미 찾아둔 것

**같은 걸 다시 찾지 마세요.** 여기 없는 것을 찾는 게 이 노트북의 목적입니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 프로파일러 발견 요약
# ═══════════════════════════════════════════════════════════
for f in AUTO.get("table_findings", []):
    print(f"🔴 [테이블] {f['detector']}: {f['message']}")

# 컬럼 그룹 — "전부 해당"은 그룹의 성질, "일부만"이 진짜 이상치
for g in AUTO.get("group_findings", []):
    print(f"\n▪ 그룹 {g['group']} ({g['size']}개)")
    for t in g.get("traits", []):
        print(f"    ─ {t['detector']}: 전부 해당 → 그룹 성질 (조치 불필요)")
    for e in g.get("exceptions", []):
        print(f"    ⚠️ {e['detector']}: {e['ratio']:.0%} — {', '.join(e['columns'][:5])}")
    if g.get("value_shapes"):
        print(f"    ⚠️ 값 표현 {len(g['value_shapes'])}가지로 갈림")

# 실제 값이 절반 미만인 컬럼 — 답변 정책이 필요한 컬럼
thin = [(c, e["values_present"], e["total"]) for c, e in AUTO["columns"].items()
        if e["total"] and e["values_present"] / e["total"] < 0.5]
print("\n📉 실제 값 < 50% 컬럼")
display(pd.DataFrame(sorted(thin, key=lambda r: r[1]),
                     columns=["column", "values_present", "total"]).head(15))

# 결측 여부 판정이 필요한 값 — <domain>.yaml 의 missing_semantics 에 선언
pend = [(c, j["value"], j["count"], j["hint"])
        for c, e in AUTO["columns"].items() for j in e.get("judgment_needed", [])]
if pend:
    print("\n❓ 결측 판정 필요 (EDA_GUIDE §3 🕳️)")
    display(pd.DataFrame(sorted(pend, key=lambda r: -r[2]),
                         columns=["column", "value", "count", "hint"]).head(10))

---

# 🧬 엔티티 탐색 — 무엇을 노드로 세울 것인가

테이블이 **한 덩어리로 비정규화**돼 있습니다. 온톨로지를 그리려면
어떤 컬럼이 **독립 개체(노드)** 이고, 어떤 컬럼이 **하위클래스 축**이고,
어떤 컬럼이 그냥 **속성**인지 갈라야 합니다.

**기계는 후보만 좁히고 판정은 사람이 합니다.** 카디널리티만으로는
`판매여부(판매중/판매완료)` 와 `수탁사(18개 기관)` 를 구분할 수 없습니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1) 후보 좁히기
# ═══════════════════════════════════════════════════════════
import re

PK = "itm_no"            # ← 담당 테이블의 주 식별자로 변경
N_ENTITY = q(f"SELECT COUNT(DISTINCT {PK}) AS n FROM {DOMAIN}")["n"][0]

FLAG_RE = re.compile(r"_yn$|여부$")


def entity_scan():
    """재사용도·결측·형태로 1차 분류. '🟢 후보'만 사람이 판정하면 된다."""
    rows = []
    for c, e in AUTO["columns"].items():
        d, miss = e["distinct_count"], 1 - e["values_present"] / TOTAL
        k = e["korean_name"] or ""
        if d <= 1 or miss > 0.9:
            hint = "제외"
        elif d >= N_ENTITY * 0.9:
            hint = "식별자/고유명"
        elif FLAG_RE.search(c) or FLAG_RE.search(k) or d == 2:
            hint = "플래그"
        elif e["kind"] == "numeric" and d > 200:
            hint = "측정값"
        else:
            hint = "🟢 후보"
        rows.append(dict(column=c, 한글명=k[:18], kind=e["kind"], distinct=d,
                         재사용도=round(N_ENTITY / d) if d else 0,
                         결측률=round(miss, 2), 판정후보=hint))
    return pd.DataFrame(rows).sort_values(["판정후보", "distinct"])


scan = entity_scan()
print(f"주 식별자 {PK} 기준 개체 {N_ENTITY:,}개\n")
print(scan["판정후보"].value_counts().to_string())
display(scan.query("판정후보 == '🟢 후보'"))

## 후보 검증 도구

판정 전에 **데이터로 확인**합니다. 아래 4개는 담당 테이블 어디에나 쓸 수 있습니다.

| 함수 | 무엇을 보나 | 왜 |
| :--- | :--- | :--- |
| `label_coverage()` | 코드 컬럼에 **사람이 읽을 이름**이 있는가 | 이름이 없으면 노드는 서도 부를 수가 없음 |
| `axis_purity()` | 범주형 컬럼이 **단일 축인가** | 축이 섞여 있으면 하위클래스로 못 씀 |
| `orthogonality()` | 두 축이 **직교하는가** | 직교면 하위클래스가 아니라 별도 속성 |
| `cardinality()` | 주 노드와 **몇 대 몇**인가 | `N:M` 이면 별도 관계 필요 |

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2) 후보 검증 도구
# ═══════════════════════════════════════════════════════════

def label_coverage(code_col, name_like=("nm", "name", "desc")):
    """코드 컬럼과 짝이 되는 이름 컬럼이 테이블에 있는지 찾는다.
    없으면 그 엔티티는 '코드만 있고 이름이 없는' 상태 — 질의어로 못 부른다."""
    stem = re.sub(r"_(cd|no|code|id)$", "", code_col.lower())
    cand = [c for c in AUTO["columns"]
            if any(c.lower().endswith(s) for s in name_like)
            and (stem[:4] in c.lower() or c.lower()[:4] in stem)]
    n = q(f'SELECT COUNT(DISTINCT "{code_col}") AS n FROM {DOMAIN}')["n"][0]
    return dict(코드컬럼=code_col, 값종류=n, 이름컬럼후보=cand or "❌ 없음")


def axis_purity(cat_col, text_col, markers):
    """범주형 컬럼이 단일 축인지 검사.
    각 범주값에 대해 text_col 안의 마커 출현을 세고, 한 범주값이 여러 마커에
    걸치면 그 컬럼은 여러 축이 섞인 것이다 (→ 하위클래스로 쓸 수 없음)."""
    sel = ", ".join(
        f"""COUNT(DISTINCT CASE WHEN "{text_col}" LIKE '%{m}%' THEN {PK} END) AS "{m}" """
        for m in markers)
    df = q(f"""SELECT TRIM(CAST("{cat_col}" AS TEXT)) AS 값,
                      COUNT(DISTINCT {PK}) AS 개체수, {sel}
               FROM {DOMAIN} GROUP BY 1 ORDER BY 2 DESC""")
    hit = df[list(markers)].gt(0).sum(axis=1)
    df.insert(2, "걸친마커수", hit)
    return df


def orthogonality(col_a, col_b):
    """두 분류 축이 직교하는지 교차표로 확인.
    한쪽 값이 다른 쪽 여러 값에 골고루 퍼지면 직교 → 별도 속성으로 둬야 한다."""
    return q(f"""SELECT TRIM(CAST("{col_a}" AS TEXT)) AS a,
                        TRIM(CAST("{col_b}" AS TEXT)) AS b,
                        COUNT(DISTINCT {PK}) AS n
                 FROM {DOMAIN} GROUP BY 1,2""").pivot(
        index="a", columns="b", values="n").fillna(0).astype(int)


def cardinality(col):
    """주 노드 : 대상 카디널리티. 개체 하나가 값을 여러 개 가지면 N:M."""
    a = q(f'SELECT COUNT(DISTINCT {PK}) AS n FROM {DOMAIN} WHERE "{col}" IS NOT NULL')["n"][0]
    b = q(f'SELECT COUNT(DISTINCT TRIM(CAST("{col}" AS TEXT))) AS n FROM {DOMAIN}')["n"][0]
    multi = q(f"""SELECT COUNT(*) AS n FROM (
                    SELECT {PK} FROM {DOMAIN} WHERE "{col}" IS NOT NULL
                    GROUP BY {PK} HAVING COUNT(DISTINCT TRIM(CAST("{col}" AS TEXT))) > 1)""")["n"][0]
    return dict(컬럼=col, 개체수=a, 값종류=b, 관계=f"{a:,} : {b:,}",
                다중값=multi, 유형="N:M" if multi else "N:1")


print("검증 도구: label_coverage() axis_purity() orthogonality() cardinality()")

### 실행 — 후보를 하나씩 검증

아래는 공모펀드 사례입니다. **담당 컬럼으로 바꿔 돌려보세요.**

In [ ]:
# ── ① 코드 컬럼에 이름이 있는가 ────────────────────────────
display(pd.DataFrame([label_coverage(c) for c in
                      ["or_co_xtn_itt_cd", "trusc_xtn_itt_cd", "mtco_itm_no"]]))

# ── ② 범주형 컬럼이 단일 축인가 ────────────────────────────
# '걸친마커수'가 2 이상인 행이 있으면 그 컬럼은 여러 축이 섞인 것
purity = axis_purity("or_attr_desc", "itm_nm", ["주식", "채권", "재간접", "파생"])
display(purity)
mixed = purity[purity["걸친마커수"] >= 2]
print(f"→ 여러 마커에 걸친 범주값 {len(mixed)}개"
      f"{' — 단일 축 아님 ⚠️' if len(mixed) else ' — 단일 축 ✅'}")

In [ ]:
# ── ③ 축이 섞였다면 규칙 보완의 효과를 측정 ─────────────────
def rule_impact(label, base_where, extended_where):
    """규칙을 보완했을 때 커버리지가 얼마나 늘어나는지 실측."""
    a = q(f"SELECT COUNT(DISTINCT {PK}) AS n FROM {DOMAIN} WHERE {base_where}")["n"][0]
    b = q(f"SELECT COUNT(DISTINCT {PK}) AS n FROM {DOMAIN} WHERE {extended_where}")["n"][0]
    return dict(질의=label, 기존=a, 보완=b, 누락됐던=b - a,
                누락률=f"{(b - a) / b:.1%}" if b else "-")


display(pd.DataFrame([
    rule_impact("주식형 펀드",
                "TRIM(or_attr_desc)='주식형'",
                "TRIM(or_attr_desc)='주식형' OR "
                "(TRIM(or_attr_desc) IN ('재간접','06') AND itm_nm LIKE '%(주식%')"),
    rule_impact("채권형 펀드",
                "TRIM(or_attr_desc)='채권형'",
                "TRIM(or_attr_desc)='채권형' OR "
                "(TRIM(or_attr_desc) IN ('재간접','06') AND itm_nm LIKE '%(채권%')"),
]))

In [ ]:
# ── ④ 직교 여부 · 카디널리티 ───────────────────────────────
# 재간접(운용구조)이 자산군과 직교하면 하위클래스가 아니라 별도 속성이어야 한다
print("자산군 × 재간접 여부")
display(q(f"""SELECT TRIM(or_attr_desc) AS 운용속성,
   COUNT(DISTINCT CASE WHEN itm_nm LIKE '%재간접%' THEN {PK} END) AS 재간접,
   COUNT(DISTINCT CASE WHEN itm_nm NOT LIKE '%재간접%' THEN {PK} END) AS 일반
   FROM {DOMAIN} GROUP BY 1 ORDER BY 2 DESC LIMIT 8"""))

print("\n카디널리티")
display(pd.DataFrame([cardinality(c) for c in
                      ["or_co_xtn_itt_cd", "trusc_xtn_itt_cd", "bmrk_nm", "prfd_attr_cd"]]))

## 판정

검증 결과를 놓고 후보마다 3가지를 물어봅니다.

| # | 질문 | 판정 근거가 되는 도구 |
| :-: | :--- | :--- |
| **Q1** | 이 값이 **자체 속성**을 가질 수 있는가? | 도메인 지식 |
| **Q2** | **다른 상품군에도** 등장하는가? | `cross()` (§3 질문 5) |
| **Q3** | 사용자가 이 값을 **주어로** 말하는가? | 예상 질의(§4) |

- **2개 이상 예** → 🟦 **실체(Entity)** — 독립 노드
- **1개** → 🟨 **분류값** — 하위클래스 축이거나 코드 목록
- **0개** → ⬜ **속성** — 주 노드에 붙임

> ⚠️ **유형(주식형·채권형)은 개체가 아니라 하위클래스입니다.**
> 금융 표준 온톨로지 FIBO 에서 `EquityFund` 는 `owl:Class` 이지 개체가 아닙니다.
> 반면 클래스(종류형)는 `FundShareClassUnit` 이라는 **독립 개체**입니다.
> → `docs/research/notes/FIBO_funds-model.md` (확인 수준: 부분 확인)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3) 판정 기록
# ═══════════════════════════════════════════════════════════
ENTITIES = {
    # "EntityName": dict(kind="entity", source="컬럼명", count=0, label="이름컬럼 or None",
    #                    relation="A -rel-> B", note="Q1/Q2/Q3 판정 근거"),
}

CLASS_HIERARCHY = {
    # 유형 축 — 개체가 아니라 하위클래스
    # "Fund": {"SecuritiesFund": ["EquityFund", "BondFund"], "MoneyMarketFund": []},
}

DERIVATION_RULES = {
    # 컬럼 → 엔티티/클래스 유도. 축이 섞였으면 여기서 분해한다.
    # "assetClass": dict(축="...", 규칙="...", 판정률="..."),
}

ATTRIBUTES = {
    # "식별": [...], "이름": [...], "성과": [...],
}

print(f"엔티티 {len(ENTITIES)} · 클래스축 {len(CLASS_HIERARCHY)} "
      f"· 유도규칙 {len(DERIVATION_RULES)} · 속성그룹 {len(ATTRIBUTES)}")
if ENTITIES and not CLASS_HIERARCHY:
    print("⚠️ 유형(주식형·채권형 등)을 ENTITIES 에 넣지 않았는지 확인 — 유형은 CLASS_HIERARCHY 로")

## 배정 검증 → yaml 내보내기

**전 컬럼이 배정됐는지** 확인하고 `<domain>.yaml` 로 내보냅니다.
이 3블록이 `ontology.ttl` 생성 입력이 됩니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4) 배정 검증 + yaml 내보내기 (병합)
# ═══════════════════════════════════════════════════════════
assigned = ({v.get("source") for v in ENTITIES.values()}
            | {c for cols_ in ATTRIBUTES.values() for c in cols_})
unassigned = [c for c in AUTO["columns"] if c not in assigned]

if ENTITIES:
    display(pd.DataFrame([
        dict(엔티티=k, 종류=v.get("kind"), 개수=v.get("count"), 출처=v.get("source"),
             레이블=v.get("label"), 관계=v.get("relation"))
        for k, v in ENTITIES.items()]))

print(f"배정 {len(assigned)} / 전체 {len(AUTO['columns'])}컬럼")
if unassigned:
    print(f"⚠️ 미배정 {len(unassigned)}개: {unassigned}")

if ENTITIES or CLASS_HIERARCHY or DERIVATION_RULES:
    DECL.setdefault("entities", {}).update(ENTITIES)
    DECL.setdefault("class_hierarchy", {}).update(CLASS_HIERARCHY)
    DECL.setdefault("derivation_rules", {}).update(DERIVATION_RULES)
    with DECL_PATH.open("w", encoding="utf-8") as fp:
        fp.write(f"# {DOMAIN} — 사람의 판단을 담는 파일\n")
        fp.write("# 기계적 사실은 *.auto.yaml 에 있습니다. 여기엔 판정과 규칙만 씁니다.\n")
        fp.write("# 런타임 가드레일이 이 파일을 읽습니다.\n\n")
        yaml.safe_dump(DECL, fp, allow_unicode=True, sort_keys=False, width=100)
    print(f"\n✅ {DECL_PATH.relative_to(ROOT)} — 블록 {list(DECL.keys())}")

---

# §3 — 답해야 할 질문 11개

각 절의 결론을 **`docs/eda/<domain>_notes.md`** 에 옮겨 적으세요.
셀은 자유롭게 추가하세요. 아래 코드는 출발점일 뿐입니다.

---

## 🔹 개체 (Entity)

### Q1. 내 상품군에서 "상품 하나"를 식별하는 것은 무엇인가?

> 자명해 보여도 확인하세요. 공모펀드는 `std_itm_no` 가 PK가 아니었습니다 —
> 같은 펀드가 속성코드별로 최대 16행입니다.

**메모:** _(여기에 답을 적으세요)_

In [ ]:
# 식별자 후보 컬럼들의 유일성 확인
for c in COLS:
    if any(k in c.lower() for k in ("no", "cd", "id", "isin")):
        n = conn.execute(f'SELECT COUNT(DISTINCT "{c}") FROM {DOMAIN}').fetchone()[0]
        if n > 100:
            mark = "  ✅ 유일" if n == TOTAL else f"  ⚠️ 중복 (평균 {TOTAL/n:.1f}배)"
            print(f"{c:<28} distinct {n:>7,} / {TOTAL:,}{mark}")

### Q2. 내 테이블에 "상품이 아닌 개체"가 섞여 있는가?

> 국내ETF마스터에는 ETN이 30.7% 들어 있습니다. 같은 개체로 볼 것인가, 나눌 것인가?

**메모:**

In [ ]:
# 상품 유형을 가르는 컬럼이 있는지 (저카디널리티 범주형 훑기)
for c in COLS:
    n = conn.execute(f'SELECT COUNT(DISTINCT TRIM(CAST("{c}" AS TEXT))) FROM {DOMAIN}').fetchone()[0]
    if 2 <= n <= 8:
        print(f"── {c}  ({kor(c)})")
        print(vals(c, limit=8).to_string(index=False), "\n")

---

## 🔹 관계 (Relation)

### Q3. 내 상품군의 상품은 무엇과 연결되는가?

> 후보: 운용사 · 발행사 · 기초지수 · 벤치마크 · 상장시장 · 기초자산 · 통화 · 구성종목 …
> **어떤 컬럼으로** 그 연결이 들어 있는지 함께 적어주세요.

**메모:**

| 연결 대상 | 우리 테이블의 컬럼 | 비고 |
| :--- | :--- | :--- |
|  |  |  |

In [ ]:
# 한글 컬럼명으로 "연결" 성격의 컬럼 훑기
meta = q("SELECT column_name, korean_name FROM schema_metadata WHERE table_name=?", (DOMAIN,))
display(meta[meta["korean_name"].str.contains(
    "사|회사|운용|발행|지수|벤치마크|시장|통화|국가|지역|자산", na=False)])

### Q4. 그 연결 대상 중 다른 상품군에도 등장할 것은? ★

> **워크샵의 핵심 재료입니다. 여기서 공통 축이 *발견*됩니다.**
> 예: 채권 발행사와 ETF 운용사가 같은 금융그룹일 수 있습니다.

**메모:**

### Q5. 우리 테이블의 그 값이 다른 테이블과 같은 표기를 쓰고 있을까?

> 이미 확인된 불일치는 `EDA_GUIDE.md` §5-A 에 있습니다 — **거기 없는 것**을 찾아주세요.

**메모:**

In [ ]:
# cross() 로 두 테이블의 같은 축 값이 겹치는지 확인
# 예시 — 담당 테이블에 맞게 바꿔 쓰세요
# cross("cu_fund_mgmt_co", "overseas_etfs", "cu_fund_mgmt_co")
# cross("wu_inv_rgn",      "public_funds",  "fd_ivst_rgn_desc")
# cross("wu_inv_ast_type", "overseas_etfs", "wu_inv_ast_type")

---

## 🔹 분류 (Classification)

### Q6. 내 상품군을 분류하는 방식이 몇 가지이고, 무엇이 1차 분류인가?

> 자산군? 운용전략? 투자지역? 위험등급? 여러 축이 있다면 서로 직교하는지 겹치는지.

**메모:**

### Q7. 그 분류에 계층이 있는가?

> 예: 일본 ⊂ 아시아 ⊂ 글로벌. 계층이 있으면 온톨로지의 `rdfs:subClassOf` 가 실제로 값어치를 합니다.

**메모:**

In [ ]:
# 두 분류 축이 직교하는지 교차표로 확인
# 예시 — 담당 테이블에 맞게
# display(q(f"""SELECT TRIM(wu_inv_ast_type) AS 자산군, TRIM(wu_inv_rgn) AS 지역, COUNT(*) AS n
#               FROM {DOMAIN} GROUP BY 1,2 ORDER BY n DESC LIMIT 30""")
#           .pivot(index="자산군", columns="지역", values="n").fillna(0).astype(int))

---

## 🔹 고유 개념 & 한계

### Q8. 내 상품군에만 있고 다른 상품군엔 없는 개념은?

> 채권: 만기·듀레이션·신용등급 / ETF: 괴리율·추적오차·합성 vs 실물 / 펀드: 클래스 체계·재간접
> → 온톨로지에서 공통 상위 클래스가 아니라 **하위 클래스 고유 속성**이 됩니다.

**메모:**

### Q9. 사용자가 물을 법한데 지금 데이터로 답할 수 없는 것은?

> 이게 그대로 `확인할 수 없음` 응답 정책이 됩니다.

**메모:**

### Q10. 비전공자 팀원이 이 상품군 질의를 이해하려면 최소한 알아야 할 도메인 지식은?

> 워크샵에서 다른 트랙 담당자에게 설명한다고 생각하고 적어주세요.

**메모:**

In [ ]:
# Q9 재료 — 답변 불가 컬럼 목록 (실제 값이 10% 미만)
rows = [(c, e["values_present"], e["total"], e["korean_name"])
        for c, e in AUTO["columns"].items() if e["values_present"] < e["total"] * 0.1]
display(pd.DataFrame(sorted(rows, key=lambda r: r[1]),
                     columns=["column", "values_present", "total", "korean_name"]))

---

## 🔹 함정

### Q11. §8에 없는 새로운 데이터 함정을 발견했다면?

> **재현 SQL + 영향 건수**를 함께 적어주세요.
>
> 💡 그게 **다른 테이블에도 기계적으로 돌릴 수 있는 패턴**이면,
> `scripts/profile_table.py` 에 탐지기로 추가해 PR 보내주세요 (`EDA_GUIDE.md` §2).
> 한 사람의 발견이 4개 테이블 전체에 자동 적용됩니다.

**메모:**

In [ ]:
# 자유 탐색

---

# §4 — 예상 질의 20개

아래 `QUESTIONS` 리스트를 채우면 **검증 → jsonl 생성**이 자동으로 됩니다.
손으로 JSON 을 쓰지 마세요 — 문법 오류와 0건 반환이 여기서 바로 드러납니다.

**배분 목표**

| 항목 | 목표 |
| :--- | :--- |
| 총 문항 | 20 |
| 난이도 | 하 7 / 중 7 / 상 6 |
| `unanswerable` + `clarify` | **6개 이상** |
| `qtype` | 조건검색3 / 정보조회3 / 비교3 / 연산·순위3 / 교차상품군2 / 네거티브3 / 결측2 / 역질문1 |

> `교차상품군` 2문항은 **지금은 답이 안 나오는 게 정상**입니다.
> 그 질의가 워크샵에서 "이 관계를 온톨로지에 넣어야 한다"의 근거가 됩니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 질의 20개 — 아래 예시를 지우고 채우세요
# ═══════════════════════════════════════════════════════════
QUESTIONS = [
    dict(
        qid="ETF-D-001", difficulty="중", qtype="연산·순위", expected_behavior="answer",
        question="국내 상장 ETF 중 최근 1년 수익률이 가장 높은 채권형 ETF 3개를 알려줘.",
        gold_sql="""SELECT TRIM(pd_nm) AS pd_nm, du_er_1y
                    FROM domestic_etfs
                    WHERE pd_grp_no='ETF' AND TRIM(wu_inv_ast_type)='채권'
                      AND du_er_1y IS NOT NULL
                    ORDER BY du_er_1y DESC LIMIT 3""",
        must_include=["ETF명 3개", "1년 수익률 수치"],
        must_not_include=["수익률 전망", "매수 추천"],
        source_columns=["pd_grp_no", "wu_inv_ast_type", "du_er_1y"],
        note="pd_grp_no 필터 없으면 ETN이 섞임. du_er_1y 결측 20.6% / 크로스체크: ",
    ),
    dict(
        qid="ETF-D-018", difficulty="상", qtype="네거티브(미존재)", expected_behavior="unanswerable",
        question="KODEX AI 로봇 ETF의 총보수가 얼마야?",
        gold_sql=None,
        must_include=["확인할 수 없음", "해당 종목 없음"],
        must_not_include=["0.4%", "총보수는"],
        source_columns=[],
        note="2026-07-11 기준 미존재 상품. 멘토 경고 사항 직결. / 크로스체크: ",
    ),
    dict(
        qid="ETF-D-020", difficulty="중", qtype="역질문필요", expected_behavior="clarify",
        question="안전한 ETF 하나 추천해줘.",
        gold_sql=None,
        must_include=["위험등급", "투자 지역", "확인이 필요"],
        must_not_include=["추천드립니다", "가장 안전한"],
        source_columns=[],
        note="조건 부족 → 역질문 분기. 과제 명세의 '조건부 안내' 요구사항. / 크로스체크: ",
    ),
]
print(f"{len(QUESTIONS)}문항 작성됨 (목표 20)")

## 검증 — DoD: *"gold_sql 을 직접 실행해 결과를 눈으로 확인"*

아래 셀이 **전 문항의 `gold_sql` 을 실제로 실행**하고 결과를 보여줍니다.
❌ 가 하나도 없어야 통과입니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 질의 검증
# ═══════════════════════════════════════════════════════════
problems = []
for item in QUESTIONS:
    qid, beh, sql = item["qid"], item["expected_behavior"], item.get("gold_sql")

    if beh == "answer":
        if not sql:
            problems.append(f"{qid}: expected_behavior=answer 인데 gold_sql 이 없음"); continue
        try:
            df = q(sql)
        except Exception as e:
            problems.append(f"{qid}: SQL 오류 — {e}"); continue
        if len(df) == 0:
            problems.append(f"{qid}: 0건 반환 — 조건을 확인하세요"); continue
        print(f"✅ {qid}  {len(df)}건 — {item['question'][:45]}")
        display(df.head(5))
    else:
        if sql:
            problems.append(f"{qid}: {beh} 인데 gold_sql 이 채워져 있음 (None 이어야 함)")
        else:
            print(f"✅ {qid}  ({beh}) — {item['question'][:45]}")

print("\n" + "═" * 60)
diff = collections.Counter(i["difficulty"] for i in QUESTIONS)
beh = collections.Counter(i["expected_behavior"] for i in QUESTIONS)
qt = collections.Counter(i["qtype"] for i in QUESTIONS)
neg = beh["unanswerable"] + beh["clarify"]

# DoD 충족 여부 — 이것도 통과해야 산출물을 낼 수 있습니다
shortfalls = []
if len(QUESTIONS) != 20:
    shortfalls.append(f"총 문항 {len(QUESTIONS)} / 20")
if (diff["하"], diff["중"], diff["상"]) != (7, 7, 6):
    shortfalls.append(f"난이도 배분 하{diff['하']}/중{diff['중']}/상{diff['상']} (목표 7/7/6)")
if neg < 6:
    shortfalls.append(f"unanswerable+clarify {neg} / 6 이상")
if any("크로스체크" in i.get("note", "") and i.get("note", "").rstrip().endswith(":")
       for i in QUESTIONS):
    shortfalls.append("크로스체크 승인자 이름이 비어 있는 문항 있음")

mark = lambda ok: "✅" if ok else "❌"
print(f"총 문항   {len(QUESTIONS):>2} / 20        {mark(len(QUESTIONS) == 20)}")
print(f"난이도    하{diff['하']} 중{diff['중']} 상{diff['상']}   (목표 하7/중7/상6) "
      f"{mark((diff['하'], diff['중'], diff['상']) == (7, 7, 6))}")
print(f"answer 외 {neg:>2} / 6 이상     {mark(neg >= 6)}")
print(f"qtype     {dict(qt)}")

if problems:
    print("\n❌ SQL 문제 — 반드시 고쳐야 합니다:")
    for p in problems:
        print("   -", p)
if shortfalls:
    print("\n⚠️  DoD 미충족:")
    for s in shortfalls:
        print("   -", s)
if not problems and not shortfalls:
    print("\n🎉 전 항목 통과 — 다음 셀에서 jsonl 을 생성하세요.")

## 산출물 생성

검증을 통과했으면 실행하세요. `eval/questions_<domain>.jsonl` 이 만들어집니다.

- **SQL 문제**가 있으면 무조건 막힙니다 (고쳐야 함)
- **DoD 미충족**(문항 수·난이도 배분 등)이면 막히되, 중간 저장이 필요하면 `ALLOW_PARTIAL = True`

> ⚠️ **2인 크로스체크가 남아 있습니다.** 다른 담당자 1명이 `gold_sql` 을 재실행해 승인하고,
> 각 문항의 `note` 끝에 승인자 이름을 남겨야 2단계 완료입니다.

In [ ]:
# ═══════════════════════════════════════════════════════════
# eval/questions_<domain>.jsonl 생성
# ═══════════════════════════════════════════════════════════
ALLOW_PARTIAL = False      # 작업 중 중간 저장이 필요하면 True

if problems:
    raise RuntimeError(f"❌ SQL 문제 {len(problems)}건 — 위 검증 셀을 먼저 해결하세요")
if shortfalls and not ALLOW_PARTIAL:
    raise RuntimeError(
        f"❌ DoD 미충족 {len(shortfalls)}건: {shortfalls}\n"
        f"   완성 후 다시 실행하거나, 중간 저장이면 ALLOW_PARTIAL = True 로 두세요"
    )
if shortfalls:
    print(f"⚠️  DoD 미충족 상태로 저장합니다 (ALLOW_PARTIAL=True): {shortfalls}\n")

out = ROOT / "eval" / f"questions_{DOMAIN}.jsonl"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as fp:
    for item in QUESTIONS:
        row = dict(item)
        if row.get("gold_sql"):
            row["gold_sql"] = " ".join(row["gold_sql"].split())   # 한 줄로 정리
        fp.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"✅ {out.relative_to(ROOT)} — {len(QUESTIONS)}문항 저장")
print("\n다음 할 일:")
print("  1. docs/eda/<domain>_notes.md 에 §3 질문 11개의 답 정리")
print("  2. 질의 20문항 2인 크로스체크 (note 에 승인자 이름)")
print("  3. §3 질문 5·9·11 의 답을 T0(리드)에 공유 — 워크샵 안건용")
print("  4. PR: 브랜치 eda/<domain>  (노트북은 커밋되지 않습니다)")